In [2]:
!pip install -q torch torchvision tqdm scikit-learn seaborn Pillow

In [3]:
import os, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision import transforms, models
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, accuracy_score)
warnings.filterwarnings('ignore')
torch.manual_seed(42)
GPU = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
#Early config for throughout the code 
#local directories
mangroveDir   = Path(r'C:\Users\FSOS\Music\zow\man')
nonmangroveDir  = Path(r'C:\Users\FSOS\Music\zow\non_tiles')
predictionFolderDir = Path(r'C:\Users\FSOS\Music\zow\predict')
bestMSaved = 'best_result_binary.pth'
ftype = {'.jpg', '.jpeg', '.png'}

In [ ]:
#Counting and checking class imbalance in trainng and testing data
mangroveImage = sorted(file for file in mangroveDir.rglob('*')  if file.suffix.lower() in ftype)
nonmangroveImage = sorted(file for file in nonmangroveDir.rglob('*') if file.suffix.lower() in ftype)
ratio = len(mangroveImage) / max(len(nonmangroveImage), 1) #ratio of mangrove to nonmangrove
if ratio < 0.5 or ratio > 2.0:
    print('There is an imbalance')


Total Mangrove images: 20400
Total Non mangrove images: 0
Ratio of mangrove to Non mangrove: 20400.00
There is an imbalance


In [6]:
class MangroveDataset(Dataset):
    def __init__(self, paths: list[Path], labels: list[int], transform=None):
        self.paths     = paths #file paths man/non
        self.labels    = labels #if it is a mangrove or not
        self.transform = transform #image transformation
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB') #force converting all images to 3 band rgb
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [7]:
#training augmentation on the data using the standard deviation and mean of efficentnet
training = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

#validation augmentation using the standard deviation and mean of efficentnet
validation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

inferenceTransform = validation 

In [8]:
#shuffling data
allPaths  = mangroveImage + nonmangroveImage
totalLabels = [1] * len(mangroveImage) + [0] * len(nonmangroveImage) #encoding #1 as mangrove and #2 and non mangrovep
rng = np.random.default_rng(42)
idx = rng.permutation(len(allPaths)) #shuffling all the data
allPaths  = [allPaths[i]  for i in idx]
totalLabels = [totalLabels[i] for i in idx]

In [9]:
#percentage splitting of training and validation
n = len(allPaths)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

#splitting the dataset into 3
trainPaths, train_labels = allPaths[:n_train], totalLabels[:n_train]
valPaths, val_labels = allPaths[n_train:n_train+n_val], totalLabels[n_train:n_train+n_val]
testPaths, test_labels = allPaths[n_train+n_val:], totalLabels[n_train+n_val:]

#applying transformations on the data
train = MangroveDataset(trainPaths, train_labels, training)
validation = MangroveDataset(valPaths, val_labels, validation)
test  = MangroveDataset(testPaths, test_labels, validation)

In [10]:
#counting weight total samples and assignment weight to the class
counts = np.bincount(train_labels)
cls_weights = 1.0 / counts
sample_wts = [cls_weights[l] for l in train_labels]

#using sampler to correct the balance of the classes
sampler = WeightedRandomSampler(sample_wts, len(sample_wts), replacement=True)

#training, validation and testing with samples 
train_loader = DataLoader(train, batch_size=32, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(validation, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test,  batch_size=32, shuffle=False,num_workers=0, pin_memory=True)

In [11]:
def build_model(freeze_backbone: bool = True) -> nn.Module:
    model = models.efficientnet_b0(weights='IMAGENET1K_V1') #loading efficent's baseline model 
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False #freezing layers 
    in_feats = model.classifier[1].in_features
    #Building a custom binary CNN model on efficent net
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_feats, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 2),)
    return model

model = build_model(freeze_backbone=True).to(GPU) #Utilising GPU if avalaible 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())


In [12]:
def trainingEpoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = correct = total = 0 #eval metrics 
    for images, labels in tqdm(loader, leave=False, desc='  train'):
        images, labels = images.to(GPU), labels.to(GPU)
        
        optimizer.zero_grad() #after every batch, the gradiant is reset
        outputs = model(images) 
        loss = criterion(outputs, labels) #calculating cross-entropy loss
        
        loss.backward() #calculate gradiant
        optimizer.step() #updating the weights of the model using the gradiant
        
        #calculating accuracy 
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


def validationEpoch(model, loader, criterion): 
    model.eval()
    total_loss = correct = total = 0
    #disabling the gradiants
    with torch.no_grad(): 
        for images, labels in loader:
            images, labels = images.to(GPU), labels.to(GPU)
           
            outputs = model(images) 
            loss    = criterion(outputs, labels)

            #calculating accuracy 
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total


def runningTraining(model, train_loader, val_loader, epochs, lr, label=''):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr) #using adam as the optimizer for the binary classifcation / can use AdamW
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, verbose=True) #this reduces learning rate as there is no improvements

    topVal  = 0.0
    pCounter  = 0
    statHistory  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        #training and validating each epoch that occurs
        tr_loss, tr_acc = trainingEpoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = validationEpoch(model, val_loader, criterion)
        scheduler.step(vl_loss) #adjusting the learning rate


        statHistory['train_loss'].append(tr_loss)
        statHistory['train_acc'].append(tr_acc)
        statHistory['val_loss'].append(vl_loss)
        statHistory['val_acc'].append(vl_acc)

        #checking for if early stopage needs to be applied
        if vl_acc > topVal:
            topVal = vl_acc
            torch.save(model.state_dict(), bestMSaved)
            pCounter = 0
        else:
            pCounter += 1
            if pCounter >= 5:
                print(f'Early stopping applied')
                break
    return statHistory

r1p1 = runningTraining(model, train_loader, val_loader,epochs=50, lr=1e-3, label='P1') #training the data


KeyboardInterrupt: 

In [ ]:
#Best weights from P1 is loaded
model.load_state_dict(torch.load(bestMSaved, map_location=GPU))

#freezoing the layers of the model and then we unfreeze the last 20 layers for fine tuning
for param in model.parameters():
    param.requires_grad = False
for param in list(model.features.parameters())[-20:]:
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

#counting the number of trainable parameters in the model
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
r2p2 = runningTraining(model, train_loader, val_loader,epochs=50, lr=1e-5, label='P2')


In [ ]:
#Evaluating the model on the test set
labelName = ['Non-Mangrove', 'Mangrove']
model.load_state_dict(torch.load(bestMSaved, map_location=GPU))
model.eval()
y_true, y_pred, y_proba = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(GPU)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1).cpu().numpy()
        preds   = probs.argmax(axis=1)
        y_proba.extend(probs[:, 1])
        y_pred.extend(preds)
        y_true.extend(labels.numpy())
y_true  = np.array(y_true)
y_pred  = np.array(y_pred)
y_proba = np.array(y_proba)

print(f'Test Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'Test AUC      : {roc_auc_score(y_true, y_proba):.4f}')
print()
print(classification_report(y_true, y_pred, target_names=labelName, digits=4))
#creaing the confusion matrix and visualising it
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=labelName, colorbar=False, ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
#This function takes the images ihe prediction folder which expects (4000x3000) and slices them into smaller 512x12 tiles. That is used to make predictions on the dataset of the images in the prediction folder
def extract_tiles(img: Image.Image,
                  tile_size: int = 512,
                  overlap: int   = 64) -> tuple[torch.Tensor, list]:
    W, H  = img.size
    step  = tile_size - overlap
    tiles, positions = [], []

    for top in range(0, max(H - tile_size + 1, 1), step):
        for left in range(0, max(W - tile_size + 1, 1), step):
            box  = (left, top,
                    min(left + tile_size, W),
                    min(top  + tile_size, H))
            tile = img.crop(box)
            if tile.size != (tile_size, tile_size):
                padded = Image.new('RGB', (tile_size, tile_size), (0, 0, 0))
                padded.paste(tile, (0, 0))
                tile = padded
            tiles.append(inferenceTransform(tile))
            positions.append(box)

    return torch.stack(tiles), positions

#this takes the file path and makes a prediction and returns the confidence of the prediction and the number of tiles that were used to make the prediction
def predict_image(image_path: str,tile_size: int  = 512,overlap: int    = 64,batch_size: int = 64,confidence_threshold: float = 0.0) -> dict:
    model.load_state_dict(torch.load(bestMSaved, map_location=GPU))
    model.eval()
    #loading the images
    img = Image.open(image_path).convert('RGB')
    tiles, _ = extract_tiles(img, tile_size, overlap)
    probability = []
    #making predictions on the dataset and calculating the confidence
    with torch.no_grad():
        for i in range(0, len(tiles), batch_size):
            batch = tiles[i:i + batch_size].to(GPU)
            probs = torch.softmax(model(batch), dim=1).cpu().numpy()
            if confidence_threshold > 0:
                probs = probs[probs.max(axis=1) >= confidence_threshold]
            if len(probs):
                probability.append(probs)
    if not probability:
        return {'prediction': 'Uncertain', 'confidence': 0.0, 'probs': [0.5, 0.5], 'n_tiles': len(tiles)}
    
    averageProb = np.concatenate(probability, axis=0).mean(axis=0)
    predicitonIndex  = int(averageProb.argmax())
    return {'prediction': labelName[predicitonIndex], 'confidence': float(averageProb[predicitonIndex]), 'probs': averageProb.tolist(), 'n_tiles': len(tiles),}




In [ ]:
#predicting in batches on the images in the prediction folder and printing the results in a table
image_files = [p for p in predictionFolderDir.rglob('*')
               if p.suffix.lower() in ftype]
if not image_files:
    print(f'No images found in {predictionFolderDir}')
else:
    print(f'Predicting on {len(image_files):,} images …\n')
    rows = []
    for img_path in tqdm(image_files):
        r = predict_image(str(img_path))
        rows.append({
            'file':       img_path.name,
            'prediction': r['prediction'],
            'confidence': f"{r['confidence']*100:.1f}%",
            'P_non_man':  f"{r['probs'][0]:.4f}",
            'P_man':      f"{r['probs'][1]:.4f}",
            'n_tiles':    r['n_tiles'],
        })

    #printing the table with predictions and confidence 
    print(f"{'File':<40} {'Prediction':<15} {'Confidence':>12}")
    print('-' * 70)
    for row in rows:
        print(f"{row['file']:<40} {row['prediction']:<15} {row['confidence']:>12}")
    counts = Counter(r['prediction'] for r in rows)
    print(f"\nSummary:")
    for cls in labelName:
        n = counts.get(cls, 0)
        print(f"  {cls:<15} {n:>5,} images  ({n/len(rows)*100:.1f}%)")